In [2]:
import torch as t
from utils import DataManager
import random
import matplotlib.pyplot as plt
import random
from probes import LRProbe, MMProbe, CCSProbe
import configparser
import json

Matplotlib created a temporary cache directory at /tmp/matplotlib-yfuedtow because the default path (/share/u/smarks/.config/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [3]:
# hyperparameters
model = 'llama-2-13b'
split = 0.8
device = 'cuda:0' if t.cuda.is_available() else 'cpu'

config = configparser.ConfigParser()
config.read('config.ini')
layer = eval(config[model]['probe_layer'])
noperiod = eval(config[model]['noperiod'])

# Reproducing generalization matrix

In [ ]:
# label 1 = the first category in each dataset's name (e.g. human in human_farmed_*).
# Training on neutral contexts and evaluating on harm contexts (and on category
# pairs never seen in training) is the generalization test: is there one shared
# human/animal direction? Note that for pairs not involving the trained categories
# (e.g. a human_farmed probe scored on companion_wild), "accuracy" measures how the
# direction *orders* the two categories rather than right/wrong answers.
train_medlies  = [
    ['human_farmed_neutral'],
    ['human_farmed_neutral', 'human_farmed_harm'],
    ['human_wild_neutral'],
    ['human_wild_neutral', 'human_wild_harm'],
]

val_datasets = [
    'human_companion_harm',
    'human_farmed_harm',
    'human_wild_harm',
    'companion_farmed_harm',
    'companion_wild_harm',
    'farmed_wild_harm',
    'human_companion_neutral',
    'human_farmed_neutral',
    'human_wild_neutral',
    'companion_farmed_neutral',
    'companion_wild_neutral',
    'farmed_wild_neutral',
]

def to_str(l):
    return '+'.join(l)

seed = random.randint(0, 10000)

In [31]:
ProbeClasses = [
    LRProbe, 
    MMProbe, 
    ]

accs = {str(probe_class) : {to_str(train_medley) : {} for train_medley in train_medlies} for probe_class in ProbeClasses}

for ProbeClass in ProbeClasses:
    for medley in train_medlies:

        # set up data
        dm = DataManager()
        for dataset in medley:
            dm.add_dataset(dataset, model, layer, split=split, seed=seed, noperiod=noperiod, center=True, device=device)
        for dataset in val_datasets:
            if dataset not in medley:
                dm.add_dataset(dataset, model, layer, split=None, noperiod=noperiod, center=True, device=device)

        # train probe
        train_acts, train_labels = dm.get('train')
        probe = ProbeClass.from_data(train_acts, train_labels, device=device)
        direction = probe.direction

        # evaluate
        for val_dataset in val_datasets:
            if val_dataset in medley:
                acts, labels = dm.data['val'][val_dataset]
                accs[str(ProbeClass)][to_str(medley)][val_dataset] = (
                    probe.pred(acts, iid=True) == labels
                ).float().mean().item()
            else:
                acts, labels = dm.data[val_dataset]
                accs[str(ProbeClass)][to_str(medley)][val_dataset] = (
                    probe.pred(acts, iid=False) == labels
                ).float().mean().item()

lr_mm_accs = accs.copy()

with open('experimental_outputs/generalization_results.json', 'r') as f:
    outs = json.load(f)

for ProbeClass in ProbeClasses:
    out = accs[str(ProbeClass)]
    out['model'] = model
    out['probe'] = ProbeClass.__str__()
    out['layer'] = layer
    out['oracle'] = False
    out['noperiod'] = noperiod
    outs.append(out)

with open('experimental_outputs/generalization_results.json', 'w') as f:
    json.dump(outs, f, indent=2)

In [ ]:
# NOTE: CCSProbe requires row-aligned negation pairs (each statement paired with its
# negation, e.g. cities/neg_cities). The animal/human datasets have no such pairs
# (harm and neutral files contain different statements and differ in length), so this
# cell only applies to the original truth datasets — skip it for animal/human runs.

ccs_medlies = [
    ['cities', 'neg_cities'],
    ['larger_than', 'smaller_than'],
]

accs = {to_str(medley) : {} for medley in ccs_medlies}

for medley in ccs_medlies:
    dm = DataManager()
    for dataset in medley:
        dm.add_dataset(dataset, model, layer, split=split, seed=seed, noperiod=noperiod, center=True, device=device)
    for dataset in val_datasets:
        if dataset not in medley:
            dm.add_dataset(dataset, model, layer, split=None, noperiod=noperiod, center=True, device=device)
    
    train_acts, train_labels = dm.data['train'][medley[0]]
    train_neg_acts, _ = dm.data['train'][medley[1]]
    probe = CCSProbe.from_data(train_acts, train_neg_acts, train_labels, device=device)

    for val_dataset in val_datasets:
        if val_dataset in medley:
            acts, labels = dm.data['val'][val_dataset]
        else:
            acts, labels = dm.data[val_dataset]
        accs[to_str(medley)][val_dataset] = (
            probe.pred(acts) == labels
        ).float().mean().item()
    
ccs_accs = accs.copy()

with open('experimental_outputs/generalization_results.json', 'r') as f:
    outs = json.load(f)

out = accs
out['model'] = model
out['probe'] = 'CCSProbe'
out['layer'] = layer
out['oracle'] = False
out['noperiod'] = noperiod
outs.append(out)

with open('experimental_outputs/generalization_results.json', 'w') as f:
    json.dump(outs, f, indent=2)

In [25]:
# get oracle probe results
oracle_accs = {str(probe_class) : {} for probe_class in ProbeClasses}
for ProbeClass in ProbeClasses:
    for dataset in val_datasets:
        dm = DataManager()
        dm.add_dataset(dataset, model, layer, split=split, noperiod=noperiod, seed=seed, device=device)
        acts, labels = dm.get('train')
        probe = ProbeClass.from_data(acts, labels, device=device)

        acts, labels = dm.data['val'][dataset]
        acc = (probe(acts, iid=True).round() == labels).float().mean().item()
        oracle_accs[str(ProbeClass)][dataset] = acc

with open('experimental_outputs/generalization_results.json', 'r') as f:
    outs = json.load(f)

for ProbeClass in ProbeClasses:
    out = oracle_accs[str(ProbeClass)]
    out['model'] = model
    out['probe'] = ProbeClass.__str__()
    out['oracle'] = True
    out['layer'] = layer
    out['noperiod'] = noperiod
    outs.append(out)

with open('experimental_outputs/generalization_results.json', 'w') as f:
    json.dump(outs, f, indent=2)

In [ ]:
models = ['llama-2-13b']  # add 'llama-2-7b' / 'llama-2-70b' once results exist for them
ProbeClasses = [LRProbe, MMProbe]  # CCS skipped: no negation pairs in the animal/human data

# load data
with open('experimental_outputs/generalization_results.json', 'r') as f:
    outs = json.load(f)

accssss = {}
averagesss = {}
for model in models:
    accsss = {}
    averagess = {}
    # get LR and MM results
    for ProbeClass in [LRProbe, MMProbe]:
        accss = {}
        averages = {}
        for train_medley in train_medlies:
            for out in outs:
                if out['model'] == model and out['probe'] == ProbeClass.__str__() and out['oracle'] == False:
                    accs = out[to_str(train_medley)]
                    break
            accss[to_str(train_medley)] = accs
            average = sum(accs.values()) / len(accs)
            averages[to_str(train_medley)] = average
        accsss[ProbeClass.__str__()] = accss
        averagess[ProbeClass.__str__()] = averages

    # # get CCS results
    # for ProbeClass in [CCSProbe]:
    #     accss = {}
    #     averages = {}
    #     for train_medley in [['cities','neg_cities'], ['larger_than', 'smaller_than']]:
    #         for out in outs:
    #             if out['model'] == model and out['probe'] == ProbeClass.__str__() and out['oracle'] == False:
    #                 accs = out[to_str(train_medley)]
    #                 break
    #         accss[to_str(train_medley)] = accs
    #         average = sum(accs.values()) / len(accs)
    #         averages[to_str(train_medley)] = average
    #     accsss[ProbeClass.__str__()] = accss
    #     averagess[ProbeClass.__str__()] = averages

    # get oracle results
    for ProbeClass in ['oracle']:
        accss = {}
        for out in outs:
            if out['model'] == model and out['oracle'] == True and out['probe'] == 'LRProbe' and all(v in out for v in val_datasets):
                accs = out
        # filter for numbers
        filtered_accs = {}
        for val_dataset in val_datasets:
            filtered_accs[val_dataset] = accs[val_dataset]
        accsss['oracle'] = filtered_accs
        averagess[ProbeClass.__str__()] = sum(filtered_accs.values()) / len(filtered_accs)

    accssss[model] = accsss
    averagesss[model] = averagess

# get few-shot results (only for datasets that have them)
with open('experimental_outputs/few_shot_results.json', 'r') as f:
    few_shot_results = json.load(f)
for model in models:
    few_shot_accs = {}
    for dataset in val_datasets:
        all_accs = [d['acc'] for d in few_shot_results if (d['dataset'] == dataset and d['model'] == model)]
        if all_accs:
            few_shot_accs[dataset] = max(all_accs)
    accssss[model]['few_shot'] = few_shot_accs


In [ ]:
# OOD accuracy per training medley (averaged over the val datasets not trained on),
# across whatever model scales are in `models` above
fig = plt.figure(figsize=(8, 4))

font = {'family' : 'Times New Roman',
        'size'   : 14}

plt.rc('font', **font)

def get_accs(full_accs, model, probe_type, medley, drop=[]):
    accs = full_accs[model][probe_type][to_str(medley)].copy()
    for d in drop:
        del accs[d]
    return accs

def get_avg(full_accs, model, probe_type, medley, drop=[]):
    accs = get_accs(full_accs, model, probe_type, medley, drop=drop)
    return sum(accs.values()) / len(accs)


styles = ['r--', 'r-', 'b--', 'b-']
for medley, style in zip(train_medlies, styles):
    plt.plot(
        [get_avg(accssss, m, 'LRProbe', medley, drop=medley) for m in models],
        style, label=to_str(medley), marker='o'
    )
plt.plot(
    [averagesss[m]['oracle'] for m in models], 'k-', label='LR on test set (oracle)', marker='o'
)

# set x ticks
plt.xticks(range(len(models)), models)

plt.ylabel('OOD accuracy')

plt.ylim(0.4, 1)
plt.legend()

plt.show()




In [ ]:
# probe-class comparison (LR vs MM) on the combined neutral+harm medleys
# (CCS and the 'likely' control from the original paper are dropped: neither has an
# animal/human analog)
plt.figure(figsize=(8, 5))

for medley, color in zip([train_medlies[1], train_medlies[3]], ['r', 'b']):
    plt.plot(
        [get_avg(accssss, m, 'LRProbe', medley, drop=medley) for m in models], f'{color}-',
        label=f'LR, {to_str(medley)}', marker='o'
    )
    plt.plot(
        [get_avg(accssss, m, 'MMProbe', medley, drop=medley) for m in models], f'{color}--',
        label=f'MM, {to_str(medley)}', marker='o'
    )
plt.plot(
    [averagesss[m]['oracle'] for m in models], 'k-', label='oracle', marker='o'
)

plt.ylim(0.4, 1)

plt.xticks(range(len(models)), models)

plt.ylabel('Accuracy')
plt.legend()

plt.show()

In [ ]:
# generalization matrix heatmap: rows = val datasets, columns = training medleys
fig, axes = plt.subplots(1, 2, figsize=(10, 8))

model = 'llama-2-13b'

def make_axis(ax, model, probe_type, medlies):
    grid = [ [None for _ in medlies] for _ in val_datasets]
    for i, dataset in enumerate(val_datasets):
        for j, medley in enumerate(medlies):
            grid[i][j] = get_accs(accssss, model, probe_type, medley)[dataset]
    ax.imshow(grid, vmin=0, vmax=1)

    for i in range(len(grid)):
        for j in range(len(grid[0])):
            ax.text(j, i, f'{round(grid[i][j] * 100):2d}', ha='center', va='center')

    ax.set_title(probe_type)
    ax.set_xticks(range(len(medlies)))
    ax.set_xticklabels([to_str(m) for m in medlies], rotation=90, fontsize=8)
    ax.set_yticks(range(len(val_datasets)))
    ax.set_yticklabels([])

make_axis(axes[0], model, 'LRProbe', train_medlies)
make_axis(axes[1], model, 'MMProbe', train_medlies)

axes[0].set_yticklabels(val_datasets, fontsize=8)

fig.tight_layout()
fig.show()
        